<sub>**The Annotated ML Handbook** · Part I: Python · by Mahir Absar Khan</sub>

# Chapter 4: Functions: reusable, testable logic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-1-python/04-python-functions.ipynb) &nbsp; [📚 Contents](../../book/README.md) &nbsp; [🌐 Web edition](https://mahir-606.github.io/annotated-ml-handbook/)

*Run the setup cell first, then read and run the cells from top to bottom. Every code cell is followed by a table explaining what each line does.*

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

A function packages a piece of logic under a name so you can reuse it, test it and reason about it in isolation. Well-named small functions are the difference between an analysis you can trust and a notebook nobody can re-run.

## 4.1 Defining and calling functions · *Beginner*

> **📖 Definitions**
>
> **Function**: a named, reusable block of code that takes inputs and can return an output. Defined with `def`, called with parentheses: `bmi(70, 1.75)`.
>
> **Parameter vs argument**: a parameter is the name in the function definition (`weight_kg`); an argument is the actual value passed when calling it (`70`).
>
> **Return value**: the result a function hands back with `return`. Without `return`, a function returns `None`.
>
> **Docstring**: a string at the start of a function, class or module that documents it; shown by `help()`.

**Anatomy of a function**

In [2]:
def bmi(weight_kg, height_m):
    """Return body-mass index (kg / m^2)."""
    value = weight_kg / height_m ** 2
    return round(value, 1)

print(bmi(70, 1.75))
print(bmi(height_m=1.6, weight_kg=55))
print(bmi.__doc__)

22.9
21.5
Return body-mass index (kg / m^2).


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>def bmi(weight_kg, height_m):</code></td><td><code>def</code>, a name, parameters in parentheses, then a colon.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;&quot;&quot;Return body-mass index (kg / m^2).&quot;&quot;&quot;</code></td><td>A <i>docstring</i>: the first string in the body documents the function.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;value = weight_kg / height_m ** 2</code></td><td>Local variable: exists only while the function runs.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return round(value, 1)</code></td><td><code>return</code> sends a value back to the caller and exits the function.</td></tr>
<tr><td><code>print(bmi(70, 1.75))</code></td><td>Call with <i>positional</i> arguments, matched by order.</td></tr>
<tr><td><code>print(bmi(height_m=1.6, weight_kg=55))</code></td><td>Call with <i>keyword</i> arguments, matched by name: order no longer matters.</td></tr>
<tr><td><code>print(bmi.__doc__)</code></td><td>The docstring is stored on the function; <code>help(bmi)</code> shows it too.</td></tr>
</table>
</details>

A function without `return` returns `None`. A function can return several values at once; it actually returns a tuple you can unpack.

**Returning several values**

In [3]:
def mean_and_std(values):
    n = len(values)
    mean = sum(values) / n
    var = sum((v - mean) ** 2 for v in values) / (n - 1)
    return mean, var ** 0.5

m, s = mean_and_std([2, 4, 4, 4, 5, 5, 7, 9])
print(f"mean={m:.3f}, sd={s:.3f}")

mean=5.000, sd=2.138


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>def mean_and_std(values):</code></td><td>Define a function <code>mean_and_std</code> that takes <code>values</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;n = len(values)</code></td><td>Number of values.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;mean = sum(values) / n</code></td><td>The arithmetic mean.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;var = sum((v - mean) ** 2 for v in values) / (n - 1)</code></td><td>Sample variance (divide by n-1).</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return mean, var ** 0.5</code></td><td>Returns the tuple (mean, std).</td></tr>
<tr><td><code>m, s = mean_and_std([2, 4, 4, 4, 5, 5, 7, 9])</code></td><td>Unpack the returned tuple into two names.</td></tr>
<tr><td><code>print(f&quot;mean={m:.3f}, sd={s:.3f}&quot;)</code></td><td>Report both, rounded to 3 decimals.</td></tr>
</table>
</details>

## 4.2 Default values, *args and **kwargs · *Intermediate*

> **📖 Definitions**
>
> **Keyword argument**: an argument passed by parameter name, as in `train(model, lr=0.01)`. **Positional arguments** are matched by order instead.
>
> ***args and **kwargs**: parameters that collect any number of extra positional arguments (into a tuple) and keyword arguments (into a dict).

**Flexible signatures**

In [4]:
def train(model, epochs=10, lr=0.001, *, verbose=False):
    return f"{model}: epochs={epochs}, lr={lr}, verbose={verbose}"

print(train("mlp"))
print(train("mlp", 50, verbose=True))

def summarize(*values, **options):
    print("values :", values)
    print("options:", options)

summarize(1, 2, 3, unit="cm", digits=2)
nums = [4, 5]
cfg = {"unit": "kg"}
summarize(*nums, **cfg)

mlp: epochs=10, lr=0.001, verbose=False
mlp: epochs=50, lr=0.001, verbose=True
values : (1, 2, 3)
options: {'unit': 'cm', 'digits': 2}
values : (4, 5)
options: {'unit': 'kg'}


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>def train(model, epochs=10, lr=0.001, *, verbose=False):</code></td><td>Parameters with <code>=</code> have defaults. After <code>*</code>, arguments are keyword-only.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return f&quot;{model}: epochs={epochs}, lr={lr}, verbose={verbose}&quot;</code></td><td>Send <code>f&quot;{model}: epochs={epochs}, lr={lr}, verbose={verbose}&quot;</code> back to the caller.</td></tr>
<tr><td><code>print(train(&quot;mlp&quot;))</code></td><td>Defaults fill in the rest.</td></tr>
<tr><td><code>print(train(&quot;mlp&quot;, 50, verbose=True))</code></td><td><code>verbose</code> MUST be passed by name because it follows <code>*</code>.</td></tr>
<tr><td><code>def summarize(*values, **options):</code></td><td><code>*values</code> collects extra positional args into a tuple; <code>**options</code> collects keyword args into a dict.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;print(&quot;values :&quot;, values)</code></td><td>The extra positional arguments, packed in a tuple.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;print(&quot;options:&quot;, options)</code></td><td>The extra keyword arguments, packed in a dict.</td></tr>
<tr><td><code>summarize(1, 2, 3, unit=&quot;cm&quot;, digits=2)</code></td><td>Three positional and two keyword arguments.</td></tr>
<tr><td><code>nums = [4, 5]</code></td><td>A list we will unpack into positional arguments.</td></tr>
<tr><td><code>cfg = {&quot;unit&quot;: &quot;kg&quot;}</code></td><td>A dict we will unpack into keyword arguments.</td></tr>
<tr><td><code>summarize(*nums, **cfg)</code></td><td>At call time, <code>*</code> and <code>**</code> <i>unpack</i> a list and a dict into arguments.</td></tr>
</table>
</details>

> **⚠️ Pitfall** · **The mutable default argument trap**
>
> Default values are created **once**, when the function is defined, not on each call.

**Why def f(x, history=[]) is a bug**

In [5]:
def add_bad(item, history=[]):
    history.append(item)
    return history

print(add_bad("a"))
print(add_bad("b"))

def add_good(item, history=None):
    if history is None:
        history = []
    history.append(item)
    return history

print(add_good("a"), add_good("b"))

['a']
['a', 'b']
['a'] ['b']


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>def add_bad(item, history=[]):</code></td><td>This single list is shared by every call that uses the default.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;history.append(item)</code></td><td>Add the item to the list.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return history</code></td><td>Send <code>history</code> back to the caller.</td></tr>
<tr><td><code>print(add_bad(&quot;a&quot;))</code></td><td>First call: [&#x27;a&#x27;], as expected.</td></tr>
<tr><td><code>print(add_bad(&quot;b&quot;))</code></td><td>Surprise: [&#x27;a&#x27;, &#x27;b&#x27;]. The list remembers the previous call.</td></tr>
<tr><td><code>def add_good(item, history=None):</code></td><td>Correct pattern: use None as the default...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;if history is None:</code></td><td>Run the indented block only if <code>history is None</code> is true.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;history = []</code></td><td>...and create a fresh list inside the function.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;history.append(item)</code></td><td>Add the item to the list.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return history</code></td><td>Send <code>history</code> back to the caller.</td></tr>
<tr><td><code>print(add_good(&quot;a&quot;), add_good(&quot;b&quot;))</code></td><td>Each call gets its own fresh list: [&#x27;a&#x27;] and [&#x27;b&#x27;].</td></tr>
</table>
</details>

## 4.3 Scope: where names live · *Intermediate*

> **📖 Definitions**
>
> **Scope**: the region of a program where a name is visible. Names created inside a function are **local** to it; names at the top level of a file are **global**.


Python looks names up in the order **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in (the *LEGB* rule).

**Local vs global names**

In [6]:
threshold = 0.5

def classify(p):
    label = 1 if p >= threshold else 0
    return label

def bump():
    threshold = 0.9
    return threshold

print(classify(0.7), bump(), threshold)

1 0.9 0.5


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>threshold = 0.5</code></td><td>A <i>global</i> name, defined at the top level of the script.</td></tr>
<tr><td><code>def classify(p):</code></td><td>Define a function <code>classify</code> that takes <code>p</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;label = 1 if p &gt;= threshold else 0</code></td><td>Reading a global inside a function is allowed...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return label</code></td><td>Send <code>label</code> back to the caller.</td></tr>
<tr><td><code>def bump():</code></td><td>Define a function <code>bump</code> with no arguments.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;threshold = 0.9</code></td><td>...but ASSIGNING creates a new LOCAL name; the global is untouched.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return threshold</code></td><td>Send <code>threshold</code> back to the caller.</td></tr>
<tr><td><code>print(classify(0.7), bump(), threshold)</code></td><td>1, 0.9, and the global threshold is still 0.5.</td></tr>
</table>
</details>

> **💡 Tip** · **Pass data in, return data out**
>
> Functions that read and modify globals are hard to test and reuse. Prefer functions whose output depends only on their arguments. Avoid the `global` keyword in analysis code.

## 4.4 Lambda functions and functions as values · *Intermediate*

> **📖 Definitions**
>
> **Lambda**: a small anonymous function written in one expression. *Example:* `lambda x: x ** 2`.
>
> **Higher-order function**: a function that takes another function as an argument or returns one, such as `sorted(key=...)`, `map`, `filter`.


Functions are ordinary objects: you can store them in variables, put them in lists and pass them to other functions. A `lambda` is a small anonymous function written in one expression.

**Passing functions around**

In [7]:
square = lambda x: x ** 2
print(square(7))
people = [("Ana", 31), ("Bo", 25), ("Cy", 40)]
print(sorted(people, key=lambda p: p[1]))
print(list(map(lambda x: x * 10, [1, 2, 3])))
print(list(filter(lambda x: x > 1, [0, 1, 2, 3])))

def apply_twice(f, x):
    return f(f(x))
print(apply_twice(square, 3))

49
[('Bo', 25), ('Ana', 31), ('Cy', 40)]
[10, 20, 30]
[2, 3]
81


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>square = lambda x: x ** 2</code></td><td>Equivalent to <code>def square(x): return x ** 2</code>.</td></tr>
<tr><td><code>print(square(7))</code></td><td>Call the lambda like any function: 49.</td></tr>
<tr><td><code>people = [(&quot;Ana&quot;, 31), (&quot;Bo&quot;, 25), (&quot;Cy&quot;, 40)]</code></td><td>A list of (name, age) tuples.</td></tr>
<tr><td><code>print(sorted(people, key=lambda p: p[1]))</code></td><td>Sort tuples by their second element (age).</td></tr>
<tr><td><code>print(list(map(lambda x: x * 10, [1, 2, 3])))</code></td><td><code>map</code> applies a function to every item (a comprehension is often clearer).</td></tr>
<tr><td><code>print(list(filter(lambda x: x &gt; 1, [0, 1, 2, 3])))</code></td><td><code>filter</code> keeps items where the function returns True.</td></tr>
<tr><td><code>def apply_twice(f, x):</code></td><td>A <i>higher-order</i> function: takes a function as an argument.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return f(f(x))</code></td><td>Send <code>f(f(x))</code> back to the caller.</td></tr>
<tr><td><code>print(apply_twice(square, 3))</code></td><td>square(square(3)) = 81.</td></tr>
</table>
</details>

## 4.5 Closures and decorators · *Advanced*

> **📖 Definitions**
>
> **Closure**: a function that remembers variables from the scope in which it was created.
>
> **Decorator**: a function that takes a function and returns a modified version; applied with `@name` above a definition.


A **closure** is a function that remembers variables from the scope where it was created. A **decorator** is a function that takes a function and returns an enhanced version; the `@name` syntax applies it.

**A timing decorator**

In [8]:
import time
from functools import wraps

def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed * 1000:.2f} ms")
        return result
    return wrapper

@timed
def slow_sum(n):
    return sum(i * i for i in range(n))

total = slow_sum(200_000)
print(total)

slow_sum took 7.61 ms
2666646666700000


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import time</code></td><td>Import the standard-library <code>time</code> module.</td></tr>
<tr><td><code>from functools import wraps</code></td><td><code>wraps</code> copies the original function&#x27;s name and docstring onto the wrapper.</td></tr>
<tr><td><code>def timed(func):</code></td><td>The decorator receives the function to wrap.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;@wraps(func)</code></td><td>Decorate the wrapper so it keeps <code>func</code>&#x27;s name and docstring.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;def wrapper(*args, **kwargs):</code></td><td>Accept any arguments so it can wrap any function.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;start = time.perf_counter()</code></td><td>High-resolution timer.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;result = func(*args, **kwargs)</code></td><td>Call the original function.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;elapsed = time.perf_counter() - start</code></td><td>Seconds elapsed during the call.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;print(f&quot;{func.__name__} took {elapsed * 1000:.2f} ms&quot;)</code></td><td>Report the function&#x27;s name and its run time in milliseconds.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;return result</code></td><td>Return the original result unchanged.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return wrapper</code></td><td>Send <code>wrapper</code> back to the caller.</td></tr>
<tr><td><code>@timed</code></td><td>Same as writing: slow_sum = timed(slow_sum)</td></tr>
<tr><td><code>def slow_sum(n):</code></td><td>Define a function <code>slow_sum</code> that takes <code>n</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return sum(i * i for i in range(n))</code></td><td>Send <code>sum(i * i for i in range(n))</code> back to the caller.</td></tr>
<tr><td><code>total = slow_sum(200_000)</code></td><td>Underscores in numbers are allowed for readability.</td></tr>
<tr><td><code>print(total)</code></td><td>Display the value of <code>total</code>.</td></tr>
</table>
</details>

**Caching expensive results**

In [9]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80))
print(fib.cache_info())

23416728348467685
CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from functools import lru_cache</code></td><td>A built-in decorator that memoises results.</td></tr>
<tr><td><code>@lru_cache(maxsize=None)</code></td><td>Remember every (argument -&gt; result) pair.</td></tr>
<tr><td><code>def fib(n):</code></td><td>Define a function <code>fib</code> that takes <code>n</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return n if n &lt; 2 else fib(n - 1) + fib(n - 2)</code></td><td>Send <code>n if n &lt; 2 else fib(n - 1) + fib(n - 2)</code> back to the caller.</td></tr>
<tr><td><code>print(fib(80))</code></td><td>Instant; without the cache this recursion would take years.</td></tr>
<tr><td><code>print(fib.cache_info())</code></td><td>Cache statistics: hits, misses and current size.</td></tr>
</table>
</details>

## 4.6 Generators: producing values lazily · *Intermediate*

> **📖 Definitions**
>
> **Generator**: a function containing `yield` that produces a sequence of values one at a time, pausing between them. It saves memory for large or infinite sequences.


A function containing `yield` becomes a **generator**. Each `yield` hands out one value and pauses the function until the next value is requested. Generators let you process data that does not fit in memory, and they are the idea behind Keras data pipelines.

**A mini-batch generator**

In [10]:
def batches(data, batch_size):
    for start in range(0, len(data), batch_size):
        yield data[start:start + batch_size]

for b in batches(list(range(10)), 4):
    print(b)
squares = (x * x for x in range(5))
print(sum(squares))

[0, 1, 2, 3]
[4, 5, 6, 7]
[8, 9]
30


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>def batches(data, batch_size):</code></td><td>Define a function <code>batches</code> that takes <code>data</code> and <code>batch_size</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;for start in range(0, len(data), batch_size):</code></td><td>Step through the data in chunks.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;yield data[start:start + batch_size]</code></td><td>Hand out one chunk, then pause here.</td></tr>
<tr><td><code>for b in batches(list(range(10)), 4):</code></td><td>The loop asks for values one at a time.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;print(b)</code></td><td>Display the value of <code>b</code>.</td></tr>
<tr><td><code>squares = (x * x for x in range(5))</code></td><td>A <i>generator expression</i>: like a list comprehension with ().</td></tr>
<tr><td><code>print(sum(squares))</code></td><td>Consumed lazily by sum(); no list is ever built.</td></tr>
</table>
</details>

## 4.7 Type hints and docstrings · *Intermediate*

> **📖 Definitions**
>
> **Type hint**: an annotation stating the expected type of a parameter or return value, such as `def f(x: float) -> list[float]`. Not enforced at run time.


Type hints document what a function expects and returns. Python does not enforce them at run time, but editors and tools such as `mypy` use them to catch mistakes early.

**A well-documented function**

In [11]:
def normalize(values: list[float], eps: float = 1e-12) -> list[float]:
    """Scale values to the range [0, 1].

    Parameters
    ----------
    values : list of float
        Raw measurements.
    eps : float, default 1e-12
        Guards against division by zero when all values are equal.

    Returns
    -------
    list of float
    """
    lo, hi = min(values), max(values)
    return [(v - lo) / (hi - lo + eps) for v in values]

print(normalize([10.0, 15.0, 20.0]))
print(normalize.__annotations__)

[0.0, 0.49999999999995, 0.9999999999999]
{'values': list[float], 'eps': <class 'float'>, 'return': list[float]}


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>def normalize(values: list[float], eps: float = 1e-12) -&gt; list[float]:</code></td><td>Type hints: <code>values</code> is a list of floats, <code>eps</code> a float; <code>-&gt; list[float]</code> is the return type.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;&quot;&quot;</code></td><td>This &quot;NumPy docstring&quot; layout is used by NumPy, pandas and scikit-learn.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;lo, hi = min(values), max(values)</code></td><td>Smallest and largest values.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return [(v - lo) / (hi - lo + eps) for v in values]</code></td><td>Send <code>[(v - lo) / (hi - lo + eps) for v in values]</code> back to the caller.</td></tr>
<tr><td><code>print(normalize([10.0, 15.0, 20.0]))</code></td><td>[0.0, 0.5, 1.0] (up to tiny rounding from eps).</td></tr>
<tr><td><code>print(normalize.__annotations__)</code></td><td>The hints are stored on the function.</td></tr>
</table>
</details>

> **🔬 Research practice** · **Functions make analyses reproducible**
>
> Wrap each stage of an analysis (load, clean, featurize, train, evaluate) in a function with clear inputs and outputs. You can then re-run the whole pipeline with a different seed or dataset by changing one argument, which is what reviewers ask for.

> **✍️ Try it yourself**
>
> 1. Write `def describe(values)` that returns a dict with `n`, `mean`, `min` and `max`.
> 2. Write a decorator `@count_calls` that prints how many times a function has been called.
> 3. Write a generator `def running_mean(xs)` that yields the mean of the items seen so far.
>
> *Worked solutions:* [solutions-parts-1-4.ipynb](../../solutions/solutions-parts-1-4.ipynb), section *Chapter 4*.

---

⬅️ [Chapter 3: Control flow: decisions and loops](03-python-control.ipynb) · [📚 Contents](../../book/README.md) · [Chapter 5: Classes and objects](05-python-oop.ipynb) ➡️